# Skill Extraction Pipeline (optimized)

Continues from `exploration.ipynb`. This version:
1. Writes the full ~150-skill taxonomy directly to disk (no manual file copying needed)
2. Cleans job description and resume text
3. Filters resumes to relevant categories (this dataset has no "Data Science" label, so `INFORMATION-TECHNOLOGY` + `ENGINEERING` are used as the closest proxy)
4. Builds the skill extractor and applies it to both datasets
5. Runs diagnostics so you can see extraction quality at a glance

## 1. Load your filtered data

In [1]:
import pandas as pd

ds_postings = pd.read_csv(r"C:\Users\dubey\1-Code\Resume-Job Matching NLP Tool\data\ds_postings_filtered.csv")
resumes = pd.read_csv(r"D:\JOB\Resume\Resume.csv")

print("Postings:", ds_postings.shape)
print("Resumes: ", resumes.shape)

Postings: (1187, 6)
Resumes:  (2484, 4)


## 2. Write the full skills taxonomy to disk

This writes the taxonomy directly from Python, so there's no dependency on manually downloading/copying a CSV file into the right folder. Runs every time but is instant and idempotent (safe to re-run).

In [2]:
taxonomy_csv = """skill_name,category,synonyms
python,technical,
sql,technical,structured query language
r,technical,
java,technical,
scala,technical,
c++,technical,cpp
javascript,technical,js
excel,tool,microsoft excel
tableau,tool,
power bi,tool,powerbi
looker,tool,
machine learning,technical,ml
deep learning,technical,dl
natural language processing,technical,nlp
computer vision,technical,cv
data visualization,technical,
data analysis,technical,
data engineering,technical,
data modeling,technical,
statistics,technical,statistical analysis
a/b testing,technical,ab testing
hypothesis testing,technical,
regression,technical,
classification,technical,
clustering,technical,
neural networks,technical,
tensorflow,tool,
pytorch,tool,
keras,tool,
scikit-learn,tool,sklearn
pandas,tool,
numpy,tool,
matplotlib,tool,
seaborn,tool,
spark,tool,apache spark
hadoop,tool,
kafka,tool,apache kafka
airflow,tool,apache airflow
etl,technical,extract transform load
dbt,tool,
snowflake,tool,
redshift,tool,amazon redshift
bigquery,tool,google bigquery
aws,tool,amazon web services
azure,tool,microsoft azure
gcp,tool,google cloud platform
docker,tool,
kubernetes,tool,k8s
git,tool,
github,tool,
ci/cd,technical,continuous integration
linux,technical,
bash,technical,shell scripting
rest api,technical,restful api
graphql,technical,
mongodb,tool,
postgresql,tool,postgres
mysql,tool,
nosql,technical,
data warehousing,technical,
big data,technical,
distributed systems,technical,
time series analysis,technical,
recommendation systems,technical,
llm,technical,large language models
generative ai,technical,genai
prompt engineering,technical,
huggingface,tool,hugging face
langchain,tool,
mlops,technical,
model deployment,technical,
feature engineering,technical,
data cleaning,technical,data wrangling
data pipeline,technical,
spreadsheet modeling,technical,
financial modeling,technical,
business intelligence,technical,bi
kpi tracking,technical,
dashboarding,technical,
communication,soft,
teamwork,soft,collaboration
problem solving,soft,
leadership,soft,
project management,soft,
stakeholder management,soft,
agile,technical,scrum
critical thinking,soft,
attention to detail,soft,
presentation skills,soft,
mentoring,soft,
c#,technical,csharp
c,technical,
matlab,tool,
sas,tool,
spss,tool,
apache flink,tool,flink
apache beam,tool,
terraform,tool,
jenkins,tool,
jira,tool,
confluence,tool,
salesforce,tool,
looker studio,tool,google data studio
d3.js,tool,d3
flask,tool,
django,tool,
fastapi,tool,
node.js,technical,nodejs
react,technical,reactjs
html,technical,
css,technical,
data governance,technical,
data quality,technical,
data privacy,technical,
gdpr,technical,
model monitoring,technical,
a/b test design,technical,
experimentation,technical,
causal inference,technical,
bayesian statistics,technical,
optimization,technical,
linear programming,technical,
supply chain analytics,technical,
fraud detection,technical,
anomaly detection,technical,
churn prediction,technical,
customer segmentation,technical,
market basket analysis,technical,
sentiment analysis,technical,
topic modeling,technical,
transformers,technical,
bert,technical,
gpt,technical,
opencv,tool,
image processing,technical,
speech recognition,technical,
reinforcement learning,technical,rl
xgboost,tool,
lightgbm,tool,
random forest,technical,
gradient boosting,technical,
"""

taxonomy_path = r"C:\Users\dubey\1-Code\Resume-Job Matching NLP Tool\data\skills_taxonomy.csv"
with open(taxonomy_path, "w", encoding="utf-8") as f:
    f.write(taxonomy_csv)

print("Taxonomy written to:", taxonomy_path)
print("Rows:", len(taxonomy_csv.strip().split(chr(10))) - 1)

Taxonomy written to: C:\Users\dubey\1-Code\Resume-Job Matching NLP Tool\data\skills_taxonomy.csv
Rows: 141


## 3. Text cleaning function

In [3]:
import re

def clean_text(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", text)          # fix concatenated words
    text = re.sub(r"&\w+;", " ", text)                          # HTML entities
    text = re.sub(r"<[^>]+>", " ", text)                        # HTML tags
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\+\#\.\-]", " ", text)           # strip odd characters
    text = re.sub(r"\s+", " ", text).strip()                    # collapse whitespace
    return text

ds_postings["description_clean"] = ds_postings["description"].apply(clean_text)
resumes["Resume_clean"] = resumes["Resume_str"].apply(clean_text)
print("Cleaning done.")

Cleaning done.


## 4. Filter resumes to relevant categories

This dataset spans 24 general industries with no dedicated "Data Science" label. `INFORMATION-TECHNOLOGY` and `ENGINEERING` are used as the closest proxy. This is a documented scoping decision — worth a line in your README.

In [4]:
relevant_categories = ["INFORMATION-TECHNOLOGY", "ENGINEERING"]
ds_resumes = resumes[resumes["Category"].isin(relevant_categories)].reset_index(drop=True)

print("Filtered resumes:", ds_resumes.shape)
print(ds_resumes["Category"].value_counts())

Filtered resumes: (238, 5)
Category
INFORMATION-TECHNOLOGY    120
ENGINEERING               118
Name: count, dtype: int64


## 5. Skill extractor (spaCy PhraseMatcher)

Requires: `python -m spacy download en_core_web_sm` already run in your `job` conda environment.

In [5]:
import spacy
from spacy.matcher import PhraseMatcher

class SkillExtractor:
    def __init__(self, taxonomy_path: str, spacy_model: str = "en_core_web_sm"):
        self.nlp = spacy.load(spacy_model)
        self.matcher = PhraseMatcher(self.nlp.vocab, attr="LOWER")
        self.taxonomy = pd.read_csv(taxonomy_path)
        self._build_matcher()

    def _build_matcher(self):
        self.surface_to_canonical = {}
        for _, row in self.taxonomy.iterrows():
            canonical = str(row["skill_name"]).strip().lower()
            surface_forms = [canonical]
            if pd.notna(row.get("synonyms")) and str(row["synonyms"]).strip():
                syns = [s.strip().lower() for s in str(row["synonyms"]).split(",") if s.strip()]
                surface_forms.extend(syns)
            for form in surface_forms:
                self.surface_to_canonical[form] = canonical

        patterns = [self.nlp.make_doc(form) for form in self.surface_to_canonical.keys()]
        self.matcher.add("SKILL", patterns)

    def extract(self, text: str) -> set:
        if not isinstance(text, str) or not text.strip():
            return set()
        doc = self.nlp(text)
        matches = self.matcher(doc)
        found = set()
        for match_id, start, end in matches:
            span_text = doc[start:end].text.lower()
            canonical = self.surface_to_canonical.get(span_text, span_text)
            found.add(canonical)
        return found

extractor = SkillExtractor(taxonomy_path=taxonomy_path)
print("Extractor ready. Total surface forms loaded:", len(extractor.surface_to_canonical))
print("Taxonomy rows:", len(extractor.taxonomy))

Extractor ready. Total surface forms loaded: 181
Taxonomy rows: 141


## 6. Sanity check on samples

In [6]:
sample_job_skills = extractor.extract(ds_postings["description_clean"].iloc[0])
sample_resume_skills = extractor.extract(ds_resumes["Resume_clean"].iloc[0])

print("Job title:", ds_postings["title"].iloc[0])
print("Extracted job skills:", sample_job_skills)
print()
print("Resume category:", ds_resumes["Category"].iloc[0])
print("Extracted resume skills:", sample_resume_skills)

Job title: Sr Data Engineer with Kafka
Extracted job skills: {'linux', 'optimization', 'kafka', 'snowflake', 'etl', 'sql', 'python'}

Resume category: INFORMATION-TECHNOLOGY
Extracted resume skills: {'linux', 'teamwork', 'html', 'big data'}


## 7. Apply extraction to full (filtered) datasets and save

In [7]:
ds_postings["skills"] = ds_postings["description_clean"].apply(extractor.extract)
ds_resumes["skills"] = ds_resumes["Resume_clean"].apply(extractor.extract)

ds_postings.to_csv(r"C:\Users\dubey\1-Code\Resume-Job Matching NLP Tool\data\ds_postings_with_skills.csv", index=False)
ds_resumes.to_csv(r"C:\Users\dubey\1-Code\Resume-Job Matching NLP Tool\data\ds_resumes_with_skills.csv", index=False)

print("Postings - avg skills found:", ds_postings["skills"].apply(len).mean())
print("Resumes  - avg skills found:", ds_resumes["skills"].apply(len).mean())

Postings - avg skills found: 9.177759056444819
Resumes  - avg skills found: 4.936974789915967


## 8. Diagnostics

Quick checks to confirm extraction quality before moving to the matching/scoring step.

In [8]:
print("--- Postings skill count distribution ---")
print(ds_postings["skills"].apply(len).describe())
print()
print("--- Resume skill count distribution (filtered set) ---")
print(ds_resumes["skills"].apply(len).describe())
print()
print("--- Resumes still scoring 0-1 skills ---")
low = ds_resumes[ds_resumes["skills"].apply(len) <= 1]
print(f"{len(low)} out of {len(ds_resumes)}")
if len(low) > 0:
    print()
    print(low["Resume_clean"].iloc[0][:500])

--- Postings skill count distribution ---
count    1187.000000
mean        9.177759
std         5.292418
min         0.000000
25%         5.000000
50%         9.000000
75%        12.000000
max        34.000000
Name: skills, dtype: float64

--- Resume skill count distribution (filtered set) ---
count    238.000000
mean       4.936975
std        3.569712
min        0.000000
25%        2.000000
50%        4.000000
75%        6.000000
max       20.000000
Name: skills, dtype: float64

--- Resumes still scoring 0-1 skills ---
26 out of 238

information technology specialist gs11 experience 07 2004 to current information technology specialist gs11 company name city state information technology specialist supervison project management circuit management licensed electrician alarm management alarm technician training supply quality assurance kevin l. trostle dsn 266-4800 comm. 865 336-4800 manage the assigned it communications environment with privileged access at the network level for the wing